# 02 — Flood evolution over time

**Goal**: Use the raw gridded NetCDF output (`sfincs_map.nc`) to understand how flood extent and volume evolved during the event.  
**Prerequisite**: A completed SFINCS model run (the NetCDF is written by the Fortran kernel, not the postprocessing scripts).  
**No simulation is run here** — this notebook only reads existing outputs.

---

## What is `sfincs_map.nc`?

`sfincs_map.nc` is the main gridded output file written by the SFINCS Fortran kernel. It contains the following variables:

| Variable | Dims | Description |
|----------|------|-------------|
| `zb` | (n, m) | Bed level / DEM elevation [m above datum] |
| `msk` | (n, m) | Active cell mask (0 = inactive, 1 = active, 2/3 = boundary) |
| `zs` | (time, n, m) | Water surface elevation at every output timestep [m] |
| `zsmax` | (timemax, n, m) | **Daily maximum** water surface elevation [m] |
| `corner_x`, `corner_y` | (n+1, m+1) | UTM coordinates of cell corners — used for exact area calculation |

**Time encoding**: SFINCS stores time as seconds since a reference date, not as standard datetime objects. We decode using `cftime`.

**Why `zsmax` instead of `zs`?**  
`zsmax` records the daily peak water level per cell, so it captures the worst moment of flooding each day without requiring you to load potentially thousands of hourly timesteps. The flood depth is derived as `h = max(zsmax − zb, 0)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cftime
import xarray as xr
from pathlib import Path

In [ ]:
# ── Edit these for your event ────────────────────────────────────────────────
BASE   = Path("/p/11210471-001-compass/03_Runs")
REGION = "sofala"
EVENT  = "Idai"
SCENARIO = "event_tp_era5_hourly_zarr_CF0_GTSMv41_CF0_era5_hourly_spw_IBTrACS_CF0"
# ─────────────────────────────────────────────────────────────────────────────

NC_PATH = BASE / REGION / EVENT / "sfincs" / SCENARIO / "sfincs_map.nc"

if not NC_PATH.exists():
    print(f"⚠️  Not found: {NC_PATH}")
else:
    print(f"✓  {NC_PATH}")

## Opening the dataset

We open with `decode_times=False` because SFINCS uses seconds-since-tref encoding that xarray cannot auto-decode to standard datetimes. We will decode manually with `cftime` in the next step.

In [ ]:
ds = xr.open_dataset(NC_PATH, decode_times=False)
print(ds)

In [ ]:
# Quick look at spatial dimensions and value ranges
print(f"Grid size   : {ds.dims['n']} rows × {ds.dims['m']} cols")
print(f"Time steps  : {ds.dims['time']}  (hourly zs)")
print(f"Daily steps : {ds.dims['timemax']}  (daily-max zsmax)")
print()
print(f"zb  range : {float(ds['zb'].min()):.2f} to {float(ds['zb'].max()):.2f} m")
print(f"msk values: {np.unique(ds['msk'].values)}")

## Decoding the time coordinate

The `timemax` coordinate encodes time as *seconds elapsed since the model reference date* (`tref` in `sfincs.inp`). The `units` attribute tells `cftime` how to interpret those numbers.

In [ ]:
# The units attribute looks like: "seconds since 2019-03-09 00:00:00"
print(f"timemax units    : {ds['timemax'].attrs['units']}")
print(f"timemax calendar : {ds['timemax'].attrs.get('calendar', 'standard')}")

raw_times = ds["timemax"].values
units     = ds["timemax"].attrs["units"]
calendar  = ds["timemax"].attrs.get("calendar", "standard")

# Decode to Python datetime objects
timestamps = cftime.num2pydate(raw_times, units=units, calendar=calendar)

print()
print("Daily timesteps:")
for t in timestamps:
    print(f"  {t.strftime('%Y-%m-%d %H:%M')}")

## Cell areas from corner coordinates

SFINCS stores the UTM coordinates of each cell's four corners in `corner_x` and `corner_y` (shape: n+1 × m+1). The cell area is the magnitude of the cross-product of two edge vectors:

```
v1 = east edge   (from SW to SE corner)
v2 = north edge  (from SW to NW corner)
cell_area = |v1 × v2|
```

This is more accurate than the cos(lat) approximation used for geographic rasters because SFINCS grids are in UTM (metres).

In [ ]:
cx = ds["corner_x"].values  # (n+1, m+1)
cy = ds["corner_y"].values

# Edge vectors from the SW corner
v1x = cx[:-1, 1:] - cx[:-1, :-1]  # east edge, x-component
v1y = cy[:-1, 1:] - cy[:-1, :-1]  # east edge, y-component
v2x = cx[1:,  :-1] - cx[:-1, :-1] # north edge, x-component
v2y = cy[1:,  :-1] - cy[:-1, :-1] # north edge, y-component

# |cross product| = area of each parallelogram cell
cell_area_m2 = np.abs(v1x * v2y - v1y * v2x)  # shape (n, m)

print(f"Cell area array shape : {cell_area_m2.shape}")
print(f"Min / max cell area   : {cell_area_m2.min():.1f} / {cell_area_m2.max():.1f} m²")
print(f"Typical cell size     : {np.sqrt(cell_area_m2.mean()):.1f} m  (≈ model resolution)")

## Computing flood metrics per day

For each daily timestep we:
1. Extract `zsmax[t]` — the peak water surface elevation that day
2. Compute water depth: `h = max(zsmax − zb, 0)`
3. Apply the active-cell mask (`msk > 0`)
4. Compute flood extent and volume

In [ ]:
FLOOD_THRESHOLD = 0.05  # m — ignore cells shallower than 5 cm

zb   = ds["zb"].values   # (n, m)
msk  = ds["msk"].values  # (n, m)
zsmax = ds["zsmax"].values  # (T, n, m)

active = msk > 0  # boolean mask of active cells

extent_km2  = []
volume_Mm3  = []

for t in range(zsmax.shape[0]):
    h = np.maximum(zsmax[t] - zb, 0.0)  # depth [m], clipped to 0
    flooded = active & (h > FLOOD_THRESHOLD)
    extent_km2.append((flooded * cell_area_m2).sum() / 1e6)
    volume_Mm3.append((h * flooded * cell_area_m2).sum() / 1e9)

extent_km2 = np.array(extent_km2)
volume_Mm3 = np.array(volume_Mm3)

peak_t = int(np.argmax(extent_km2))
print(f"Peak flood extent : {extent_km2[peak_t]:.1f} km²  on {timestamps[peak_t].strftime('%Y-%m-%d')}")
print(f"Peak flood volume : {volume_Mm3[peak_t]:.3f} Mm³")

## Time series plot

In [ ]:
import datetime
# Convert cftime objects to standard datetime for matplotlib
dates = [datetime.datetime(t.year, t.month, t.day, t.hour, t.minute) for t in timestamps]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.fill_between(dates, extent_km2, alpha=0.6, color="steelblue")
ax1.plot(dates, extent_km2, color="steelblue", linewidth=1.5)
ax1.axvline(dates[peak_t], color="red", linestyle="--", linewidth=1, label=f"Peak: {dates[peak_t].strftime('%d %b')}")
ax1.set_ylabel("Flood extent [km²]", fontsize=10)
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.3)
ax1.set_title(f"TC {EVENT} flood evolution (factual)", fontsize=12)

ax2.fill_between(dates, volume_Mm3, alpha=0.6, color="teal")
ax2.plot(dates, volume_Mm3, color="teal", linewidth=1.5)
ax2.axvline(dates[peak_t], color="red", linestyle="--", linewidth=1)
ax2.set_ylabel("Flood volume [Mm³]", fontsize=10)
ax2.set_xlabel("Date", fontsize=10)
ax2.grid(axis="y", alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
fig.autofmt_xdate(rotation=30)
plt.tight_layout()

## Spatial snapshot at peak flooding

Let's plot the flood depth map for the day with the largest flooded area.

In [ ]:
# Flood depth at peak timestep
h_peak = np.maximum(zsmax[peak_t] - zb, 0.0)
h_peak[~active] = np.nan  # hide inactive cells
h_peak[h_peak <= FLOOD_THRESHOLD] = np.nan  # hide dry cells

# Build x/y coordinate arrays from corner coordinates (cell centres)
x_centres = 0.5 * (cx[:-1, :-1] + cx[1:, 1:])  # (n, m)
y_centres = 0.5 * (cy[:-1, :-1] + cy[1:, 1:])

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.pcolormesh(
    cx, cy, h_peak,
    cmap="Blues", vmin=0.05, vmax=3.0,
    shading="flat",
)
plt.colorbar(sc, ax=ax, label="Flood depth [m]", shrink=0.7)

# Contour the bed level at 0 m to show approximate coastline
ax.contour(x_centres, y_centres, zb, levels=[0], colors="black", linewidths=0.6, linestyles="--")

ax.set_aspect("equal")
ax.set_title(f"Flood depth at peak — {dates[peak_t].strftime('%d %b %Y')} (UTC)", fontsize=11)
ax.set_xlabel("Easting [m UTM]")
ax.set_ylabel("Northing [m UTM]")
plt.tight_layout()

---

## Next steps

- **`03_climate_attribution.ipynb`** — compare the factual flood against counterfactual scenarios (reduced precipitation, lower sea level) to isolate the contribution of climate change.
- **`04_damage_analysis.ipynb`** — load FIAT building damage outputs to translate flood depth into economic losses.